# Probabilistic Models Benchmark Study

This notebook compares benchmarks of probabilistic forecasting models

## Models Evaluated
1.  Probabilistic Transformer
2.  Probabilistic LSTM
3.  Probabilistic DeepAR
4.  Probabilistic N-BEATS
5.  Probabilistic N-HiTS
6.  XGBoost (Quantile Regression)
7.  LightGBM (Quantile Regression)
8.  Persistence + Empirical Residual Quantiles (simplest probabilistic baseline)

## Methodology
1.  Optimization: Each model's hyperparameters are optimized
    - Deep Learning models are optimized for both distributional and quantile heads
2.  Comparison: Optimized models are evaluated on the test set
3.  Robustness: Stochastic models (Neural Networks) are run 10 times to estimate variance
4.  Metrics: CRPS (Continuous Ranked Probability Score), MAE, RMSE

In [1]:
import os
import sys
import json
import time
import gc
import warnings
import logging
from pathlib import Path
from typing import Dict, Any, List

import numpy as np
import pandas as pd
import tensorflow as tf
import optuna

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ExperimentConfig, DataConfig, ModelConfig, TrainingConfig
from data import DataPipeline
from transformations import StandardScalingTransformation
from core.trainer import Trainer
from core.evaluator import Evaluator

from models import (
    ProbabilisticTransformer, 
    ProbabilisticLSTM,
    ProbabilisticDeepAR,
    ProbabilisticNBEATS,
    ProbabilisticNHITS,
    QuantileGBDT,
    PersistenceResidual,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

RESULTS_DIR = Path(project_root) / "results" / "benchmark"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PARAMS_FILE = RESULTS_DIR / "best_params.json"
RESULTS_FILE = RESULTS_DIR / "comparison_results.json"

OUTPUT_HORIZON = 24
INPUT_WINDOW = 168 
N_TRIALS = 15 
N_REPEATS = 10 
EPOCHS = 30
PATIENCE = 5

2026-03-04 12:39:43.807638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772624383.934870 3909877 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772624383.970670 3909877 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772624384.118866 3909877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772624384.118886 3909877 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772624384.118888 3909877 computation_placer.cc:177] computation placer alr

In [2]:
print("Loading data")
data_config = DataConfig(dataset_name="BE_ENTSOE", input_window=INPUT_WINDOW, output_horizon=OUTPUT_HORIZON)
pipeline = DataPipeline(data_config)
train_df_raw, val_df_raw, test_df_raw = pipeline.get_data_splits()

def prepare_data(input_w):
    pipeline.config.input_window = input_w
    X_train, y_train = pipeline.create_sequences(train_df_raw)
    X_val, y_val = pipeline.create_sequences(val_df_raw)
    X_test, y_test = pipeline.create_sequences(test_df_raw)
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(INPUT_WINDOW)

transform = StandardScalingTransformation()
transform.fit(X_train, y_train)
X_train_s, y_train_s = transform.transform(X_train, y_train)
X_val_s, y_val_s = transform.transform(X_val, y_val)
X_test_s, y_test_s = transform.transform(X_test, y_test)

print(f"Train shape: {X_train_s.shape}")

Loading data
Train shape: (10367, 168, 28)


In [3]:
def load_best_params():
    if PARAMS_FILE.exists():
        with open(PARAMS_FILE, "r") as f:
            return json.load(f)
    return {}

def save_best_params(params_dict):
    with open(PARAMS_FILE, "w") as f:
        json.dump(params_dict, f, indent=2)

best_params_store = load_best_params()

In [4]:
# Transformer optimization
transformer_fixed_params = {
    "d_model": 224,
    "num_heads": 7,
    "num_layers": 3,
    "ff_dim": 256,
    "dropout": 0.15,
    "learning_rate": 0.0007
}

In [5]:
def run_deep_optimization(model_cls, name_prefix, head_type="johnson_su"):
    if name_prefix in best_params_store:
        print(f"{name_prefix} already optimized. Skipping.")
        return best_params_store[name_prefix]

    def objective(trial):
        # Hyperparameters
        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        layers_n = trial.suggest_int("num_layers", 1, 3)
        dropout = trial.suggest_float("dropout", 0.0, 0.3)
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        ff_dim = trial.suggest_categorical("ff_dim", [128, 256, 512])
        
        # Config
        curr_model_conf = ModelConfig(
            d_model=d_model,
            num_layers=layers_n,
            dropout=dropout,
            ff_dim=ff_dim,
            num_heads=4
        )
        curr_train_conf = TrainingConfig(
            learning_rate=lr,
            epochs=EPOCHS,
            patience=PATIENCE,
            batch_size=64
        )
        exp_conf = ExperimentConfig(
            name=f"opt_{name_prefix}",
            data_config=data_config,
            model_config=curr_model_conf,
            training_config=curr_train_conf,
            head_type=head_type
        )
        
        # Train
        tf.keras.backend.clear_session()
        model = model_cls(exp_conf)
        trainer = Trainer(exp_conf)
        
        history = trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s, verbose=0)
        
        val_loss = min(history.history['val_loss'])
        return val_loss

    print(f"Optimizing {name_prefix}")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    
    best_params_store[name_prefix] = study.best_params
    save_best_params(best_params_store)
    print(f"Best params for {name_prefix}: {study.best_params}")
    return study.best_params

In [6]:
def run_gbdt_optimization(model_type, name_prefix):
    if name_prefix in best_params_store:
        print(f"{name_prefix} already optimized. Skipping.")
        return best_params_store[name_prefix]

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0)
        }
        if model_type == "lightgbm":
            params["num_leaves"] = trial.suggest_int("num_leaves", 20, 100)
            del params["max_depth"] 
            
        model = QuantileGBDT(model_type=model_type, quantiles=[0.5], params=params)
        
        X_tr_flat = X_train_s.reshape(X_train_s.shape[0], -1)
        X_val_flat = X_val_s.reshape(X_val_s.shape[0], -1)
        
        # Subsample for speed
        if len(X_tr_flat) > 2000:
            idx = np.random.choice(len(X_tr_flat), size=2000, replace=False)
            X_tr_opt = X_tr_flat[idx]
            y_tr_opt = y_train[idx]
        else:
            X_tr_opt = X_tr_flat
            y_tr_opt = y_train
            
        model.fit(X_tr_opt, y_tr_opt)
        preds = model.predict(X_val_flat)
        
        mae = np.mean(np.abs(y_val - preds[0.5]))
        return mae

    print(f"Optimizing {name_prefix}")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)
    
    best_params_store[name_prefix] = study.best_params
    save_best_params(best_params_store)
    return study.best_params

In [7]:
# Run optimizations
dl_classes = [
    (ProbabilisticLSTM, "LSTM"),
    (ProbabilisticDeepAR, "DeepAR"),
    (ProbabilisticNBEATS, "NBEATS"),
    (ProbabilisticNHITS, "NHITS")
]

head_types = ["johnson_su", "quantile"]

for model_cls, name_base in dl_classes:
    for head in head_types:
        full_name = f"{name_base}_{head}"
        run_deep_optimization(model_cls, full_name, head_type=head)

# GBDTs
run_gbdt_optimization("xgboost", "XGBoost")
run_gbdt_optimization("lightgbm", "LightGBM")

# PersistenceResidual: no optimization needed (no hyperparameters)

LSTM_johnson_su already optimized. Skipping.
LSTM_quantile already optimized. Skipping.
DeepAR_johnson_su already optimized. Skipping.
DeepAR_quantile already optimized. Skipping.
NBEATS_johnson_su already optimized. Skipping.
NBEATS_quantile already optimized. Skipping.
NHITS_johnson_su already optimized. Skipping.
NHITS_quantile already optimized. Skipping.
XGBoost already optimized. Skipping.
LightGBM already optimized. Skipping.


{'n_estimators': 500,
 'learning_rate': 0.023491490718106106,
 'max_depth': 10,
 'subsample': 0.8049639484477037,
 'num_leaves': 20}

In [ ]:
# Comparison
import joblib

# Load existing results
if RESULTS_FILE.exists():
    try:
        final_results = pd.read_json(RESULTS_FILE).to_dict("records")
        print(f"Loaded {len(final_results)} existing results.")
    except:
        print("Could not load existing results, starting fresh.")
        final_results = []
else:
    final_results = []

existing_models = {r["Model"] for r in final_results}

# Re-evaluate models with missing PICP/MPIW/IntervalScore (e.g. from older runs)
REQUIRED_METRICS = ["PICP", "MPIW", "IntervalScore"]
def has_complete_metrics(r):
    return all(pd.notna(r.get(m, np.nan)) for m in REQUIRED_METRICS)
final_results = [r for r in final_results if has_complete_metrics(r)]
existing_models = {r["Model"] for r in final_results}

# 1. Transformer
transformer_heads = ["johnson_su", "quantile"]

def create_transformer(head_type):
    m_params = {k: v for k, v in transformer_fixed_params.items() if k != "learning_rate"}
    conf = ModelConfig(**m_params)
    tr_conf = TrainingConfig(learning_rate=transformer_fixed_params["learning_rate"], epochs=EPOCHS, patience=PATIENCE)
    exp = ExperimentConfig(
        name=f"Transformer_Base_{head_type}", 
        data_config=data_config, 
        model_config=conf, 
        training_config=tr_conf,
        head_type=head_type
    )
    model = ProbabilisticTransformer(exp)
    trainer = Trainer(exp)
    trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s, verbose=0)
    return model, exp

for head in transformer_heads:
    full_name_key = f"Transformer ({head})"
    if full_name_key in existing_models:
        print(f"Skipping {full_name_key}, already done.")
        continue
    print(f"Evaluating Transformer ({head})")
    metrics_runs = []
    for i in range(N_REPEATS):
        tf.keras.backend.clear_session()
        model, exp = create_transformer(head)
        evaluator = Evaluator(model, transform)
        m = evaluator.evaluate(X_test_s, y_test)
        metrics_runs.append(m)

    avg_metrics = {k: np.mean([x[k] for x in metrics_runs]) for k in metrics_runs[0]}
    final_results.append({"Model": f"Transformer ({head})", **avg_metrics})
    
    # Save model and results
    model_save_path = RESULTS_DIR / f"model_Transformer_{head}.pkl"
    try:
        model.save(model_save_path)
    except:
        pass
    pd.DataFrame(final_results).to_json(RESULTS_FILE)

# 2. DL models loop
dl_models = [
    ("LSTM", ProbabilisticLSTM),
    ("DeepAR", ProbabilisticDeepAR),
    ("N-BEATS", ProbabilisticNBEATS),
    ("N-HiTS", ProbabilisticNHITS)
]

for name_base, cls in dl_models:
    for head in head_types:
        full_name = f"{name_base}_{head}"
        full_name_key = f"{name_base} ({head})"
        if full_name_key in existing_models:
            print(f"Skipping {full_name_key}, already done.")
            continue
        full_name = f"{name_base}_{head}"
        print(f"Evaluating {full_name}")
        
        if full_name not in best_params_store:
            print(f"Warning: Params for {full_name} not found, skipping.")
            continue
            
        params = best_params_store[full_name]
        
        runs = []
        for i in range(N_REPEATS):
            tf.keras.backend.clear_session()
            
            mod_c = ModelConfig(
                d_model=params["d_model"], 
                num_layers=params["num_layers"], 
                dropout=params["dropout"], 
                ff_dim=params["ff_dim"]
            )
            tr_c = TrainingConfig(
                learning_rate=params["learning_rate"], 
                epochs=EPOCHS, 
                patience=PATIENCE
            )
            exp = ExperimentConfig(
                name=f"{full_name}_{i}", 
                data_config=data_config, 
                model_config=mod_c, 
                training_config=tr_c,
                head_type=head
            )
            
            model = cls(exp)
            trainer = Trainer(exp)
            trainer.train(model, X_train_s, y_train_s, X_val_s, y_val_s, verbose=0)
            
            evaluator = Evaluator(model, transform)
            m = evaluator.evaluate(X_test_s, y_test)
            runs.append(m)
            
        avg = {k: np.mean([x[k] for x in runs]) for k in runs[0]}
        final_results.append({"Model": f"{name_base} ({head})", **avg})
        
        # Save model and results
        model_save_path = RESULTS_DIR / f"model_{name_base}_{head}.pkl"
        try:
             model.save(model_save_path)
        except:
             pass
        pd.DataFrame(final_results).to_json(RESULTS_FILE)

# 3. GBDT / PersistenceResidual (simple probabilistic baseline)
def nll_from_quantiles(preds: dict, y: np.ndarray, quantiles: list) -> float:
    """Approximate NLL from quantile predictions (piecewise uniform between quantiles)."""
    q_arr = np.array(quantiles)
    eps = 1e-10
    log_pdf_sum = 0.0
    n = 0
    for i in range(y.shape[0]):
        for t in range(y.shape[1]):
            y_val = y[i, t]
            p_vals = np.array([preds[q][i, t] for q in quantiles])
            pairs = sorted(zip(q_arr, p_vals), key=lambda x: x[1])
            q_s, p_s = np.array([x[0] for x in pairs]), np.array([x[1] for x in pairs])
            if y_val <= p_s[0]:
                dq = q_s[1] - q_s[0] if len(q_s) > 1 else 0.1
                dw = max(p_s[1] - p_s[0], eps) if len(p_s) > 1 else eps
                log_pdf_sum += np.log(dq + eps) - np.log(dw)
            elif y_val >= p_s[-1]:
                dq = q_s[-1] - q_s[-2] if len(q_s) > 1 else 0.1
                dw = max(p_s[-1] - p_s[-2], eps) if len(p_s) > 1 else eps
                log_pdf_sum += np.log(dq + eps) - np.log(dw)
            else:
                idx = np.searchsorted(p_s, y_val, side="right") - 1
                idx = min(idx, len(q_s) - 2)
                dq = q_s[idx + 1] - q_s[idx]
                dw = max(p_s[idx + 1] - p_s[idx], eps)
                log_pdf_sum += np.log(dq + eps) - np.log(dw)
            n += 1
    return -log_pdf_sum / n if n > 0 else np.nan

gbdt_models = [
    ("XGBoost", "xgboost", QuantileGBDT),
    ("LightGBM", "lightgbm", QuantileGBDT),
    ("PersistenceResidual", None, PersistenceResidual),
]

# PersistenceResidual needs X in original scale (3D)
X_train_orig, _ = transform.inverse_transform(X=X_train_s, y=None)
X_test_orig, _ = transform.inverse_transform(X=X_test_s, y=None)

for name, type_key, cls in gbdt_models:
    print(f"Evaluating {name}")
    if name in existing_models:
        print(f"Skipping {name}, already done.")
        continue
    params = best_params_store.get(name, {})
    
    is_stochastic = False
    if name in ["XGBoost", "LightGBM"]:
        if params.get("subsample", 1.0) < 1.0:
            is_stochastic = True
            
    repeats = 1 
    runs = []
    
    for i in range(repeats):
        print(f"  > Repeat {i+1}/{repeats}")
        if name == "PersistenceResidual":
            model = cls(quantiles=[0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975])
            X_tr, X_te = X_train_orig, X_test_orig
        else:
            model = cls(model_type=type_key, quantiles=[0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975], params=params)
            X_tr = X_train_s.reshape(X_train_s.shape[0], -1)
            X_te = X_test_s.reshape(X_test_s.shape[0], -1)
        
        model.fit(X_tr, y_train)
        preds = model.predict(X_te)
        
        mae = np.mean(np.abs(y_test - preds[0.5]))
        rmse = np.sqrt(np.mean((y_test - preds[0.5])**2))
        
        pinball_losses = []
        for q, p_pred in preds.items():
            resid = y_test - p_pred
            loss = np.maximum(q * resid, (q - 1) * resid)
            pinball_losses.append(np.mean(loss))
        crps_proxy = np.mean(pinball_losses)
        
        # MPIW, PICP, IntervalScore (95% PI)
        q_low, q_high = preds[0.025], preds[0.975]
        covered = (y_test >= q_low) & (y_test <= q_high)
        picp = np.mean(covered)
        mpiw = np.mean(q_high - q_low)
        alpha = 0.05
        width = q_high - q_low
        lower_penalty = (2/alpha) * np.maximum(q_low - y_test, 0)
        upper_penalty = (2/alpha) * np.maximum(y_test - q_high, 0)
        interval_score = np.mean(width + lower_penalty + upper_penalty)
        
        quantiles_list = [0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975]
        nll = nll_from_quantiles(preds, y_test, quantiles_list)
        mse = np.mean((y_test - preds[0.5])**2)
        
        metrics = {"MAE": mae, "MSE": mse, "RMSE": rmse, "CRPS": crps_proxy, "PICP": picp, "MPIW": mpiw, "IntervalScore": interval_score, "NLL": nll}
        runs.append(metrics)
        
    avg = {k: np.mean([x[k] for x in runs]) for k in runs[0]}
    final_results.append({"Model": name, **avg})
    
    # Save model and results
    model_save_path = RESULTS_DIR / f"model_{name.replace(' ', '_')}.pkl"
    try:
         with open(model_save_path, 'wb') as f:
             joblib.dump(model, f)
    except:
         pass
    pd.DataFrame(final_results).to_json(RESULTS_FILE)

# Display
res_df = pd.DataFrame(final_results).sort_values("MAE")
print(res_df)
res_df.to_json(RESULTS_FILE)

Evaluating Transformer (johnson_su)


I0000 00:00:1772624389.491862 3909877 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7483 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080, pci bus id: 0000:01:00.0, compute capability: 6.1
